In [21]:
import pandas as pd

In [22]:
df = pd.read_csv("../data/raw/WineDataset.csv")
df.head()

,Title,Description,Price,Capacity,Grape,Secondary Grape Varieties,Closure,Country,Unit,Characteristics,Per bottle / case / each,Type,ABV,Region,Style,Vintage,Appellation
0,"The Guv'nor, Spain",We asked some of our most prized winemakers wo...,£9.99 per bottle,75CL,Tempranillo,NaN,Natural Cork,Spain,10.5,"Vanilla, Blackberry, Blackcurrant",per bottle,Red,ABV 14.00%,NaN,Rich & Juicy,NV,NaN
1,Bread & Butter 'Winemaker's Selection' Chardon...,This really does what it says on the tin. It’s...,£15.99 per bottle,75CL,Chardonnay,NaN,Natural Cork,USA,10.1,"Vanilla, Almond, Coconut, Green Apple, Peach, ...",per bottle,White,ABV 13.50%,California,Rich & Toasty,2021,Napa Valley
2,"Oyster Bay Sauvignon Blanc 2022, Marlborough",Oyster Bay has been an award-winning gold-stan...,£12.49 per bottle,75CL,Sauvignon Blanc,NaN,Screwcap,New Zealand,9.8,"Tropical Fruit, Gooseberry, Grapefruit, Grass,...",per bottle,White,ABV 13.00%,Marlborough,Crisp & Zesty,2022,NaN
3,Louis Latour Mâcon-Lugny 2021/22,We’ve sold this wine for thirty years – and fo...,£17.99 per bottle,75CL,Chardonnay,NaN,Natural Cork,France,10.1,"Peach, Apricot, Floral, Lemon",per bottle,White,ABV 13.50%,Burgundy,Ripe & Rounded,2022,Macon
4,Bread & Butter 'Winemaker's Selection' Pinot N...,Bread & Butter is that thing that you can coun...,£15.99 per bottle,75CL,Pinot Noir,NaN,Natural Cork,USA,10.1,"Smoke, Black Cherry, Cedar, Raspberry, Red Fruit",per bottle,Red,ABV 13.50%,California,Smooth & Mellow,2021,Napa Valley


In [23]:
df["ABV"] = pd.to_numeric(
    df["ABV"].str.extract(r'(\d*\.?\d+)')[0],
    errors="coerce"
)

In [24]:
df["Capacity"].unique()

<StringArray>
[  '75CL',   '70CL',  '750ML', '1.5LTR',    'Our', '37.5CL',  '2.25L',
  '500ML',   '50CL',  '150CL',  '300CL', '5LITRE',  '375ML']
Length: 13, dtype: str

In [25]:
df["Price"] = df["Price"].apply(lambda s: s.replace("per bottle", "").replace("per case", "").replace("each", "").strip())

In [26]:
df["Price"].unique()

<StringArray>
[  '£9.99',  '£15.99',  '£12.49',  '£17.99', '£300.00',  '£12.99',  '£80.00',
  '£13.99',   '£8.99',  '£11.99',
 ...
  '£56.00',  '£58.99', '£240.00',  '£55.00',  '£65.00',  '£60.00', '£270.00',
 '£180.00',  '£45.00',  '£40.00']
Length: 124, dtype: str

In [27]:
df["Price"] = df["Price"].astype(str).str.strip()
df[["Currency", "Price"]] = df["Price"].str.extract(r'^(\D+)\s*([\d.]+)$')
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df.head()

,Title,Description,Price,Capacity,Grape,Secondary Grape Varieties,Closure,Country,Unit,Characteristics,Per bottle / case / each,Type,ABV,Region,Style,Vintage,Appellation,Currency
0,"The Guv'nor, Spain",We asked some of our most prized winemakers wo...,9.99,75CL,Tempranillo,NaN,Natural Cork,Spain,10.5,"Vanilla, Blackberry, Blackcurrant",per bottle,Red,14.0,NaN,Rich & Juicy,NV,NaN,£
1,Bread & Butter 'Winemaker's Selection' Chardon...,This really does what it says on the tin. It’s...,15.99,75CL,Chardonnay,NaN,Natural Cork,USA,10.1,"Vanilla, Almond, Coconut, Green Apple, Peach, ...",per bottle,White,13.5,California,Rich & Toasty,2021,Napa Valley,£
2,"Oyster Bay Sauvignon Blanc 2022, Marlborough",Oyster Bay has been an award-winning gold-stan...,12.49,75CL,Sauvignon Blanc,NaN,Screwcap,New Zealand,9.8,"Tropical Fruit, Gooseberry, Grapefruit, Grass,...",per bottle,White,13.0,Marlborough,Crisp & Zesty,2022,NaN,£
3,Louis Latour Mâcon-Lugny 2021/22,We’ve sold this wine for thirty years – and fo...,17.99,75CL,Chardonnay,NaN,Natural Cork,France,10.1,"Peach, Apricot, Floral, Lemon",per bottle,White,13.5,Burgundy,Ripe & Rounded,2022,Macon,£
4,Bread & Butter 'Winemaker's Selection' Pinot N...,Bread & Butter is that thing that you can coun...,15.99,75CL,Pinot Noir,NaN,Natural Cork,USA,10.1,"Smoke, Black Cherry, Cedar, Raspberry, Red Fruit",per bottle,Red,13.5,California,Smooth & Mellow,2021,Napa Valley,£


In [28]:
df["Currency"].unique()

<StringArray>
['£']
Length: 1, dtype: str

In [29]:
extracted = df["Capacity"].str.extract(r'(?i)^\s*(\d*\.?\d+)\s*([A-Z]+)\s*$')

df["value"] = pd.to_numeric(extracted[0], errors="coerce")
df["unit"] = extracted[1].str.upper()
conversion = {
    "ML": 0.001,
    "CL": 0.01,
    "L": 1,
    "LTR": 1,
    "LITRE": 1,
    "LITRES": 1
}

df["capacity_litres"] = df["value"] * df["unit"].map(conversion)
df = df.dropna(subset=["capacity_litres"])
df["Capacity"] = df["capacity_litres"]
df.head()
df["Price"] /= df["Capacity"]

In [30]:
df = df.drop(
    columns=["value", "unit", "capacity_litres", "Currency", "Description", "Per bottle / case / each", "Capacity"]
)

In [31]:
df["Vintage"] = pd.to_numeric(df["Vintage"], errors="coerce")

In [32]:
df.head()

,Title,Price,Grape,Secondary Grape Varieties,Closure,Country,Unit,Characteristics,Type,ABV,Region,Style,Vintage,Appellation
0,"The Guv'nor, Spain",13.320000,Tempranillo,NaN,Natural Cork,Spain,10.5,"Vanilla, Blackberry, Blackcurrant",Red,14.0,NaN,Rich & Juicy,NaN,NaN
1,Bread & Butter 'Winemaker's Selection' Chardon...,21.320000,Chardonnay,NaN,Natural Cork,USA,10.1,"Vanilla, Almond, Coconut, Green Apple, Peach, ...",White,13.5,California,Rich & Toasty,2021.0,Napa Valley
2,"Oyster Bay Sauvignon Blanc 2022, Marlborough",16.653333,Sauvignon Blanc,NaN,Screwcap,New Zealand,9.8,"Tropical Fruit, Gooseberry, Grapefruit, Grass,...",White,13.0,Marlborough,Crisp & Zesty,2022.0,NaN
3,Louis Latour Mâcon-Lugny 2021/22,23.986667,Chardonnay,NaN,Natural Cork,France,10.1,"Peach, Apricot, Floral, Lemon",White,13.5,Burgundy,Ripe & Rounded,2022.0,Macon
4,Bread & Butter 'Winemaker's Selection' Pinot N...,21.320000,Pinot Noir,NaN,Natural Cork,USA,10.1,"Smoke, Black Cherry, Cedar, Raspberry, Red Fruit",Red,13.5,California,Smooth & Mellow,2021.0,Napa Valley


In [33]:
df.to_csv("../data/preprocessed/WineDatasetPAS.csv", index=False)

In [34]:
df["Characteristics"] = df["Characteristics"].apply(
    lambda x: set([c.strip() for c in str(x).split(",") if c.strip() != ""])
)
df["Style"] = df["Style"].apply(
    lambda x: set([c.strip() for c in str(x).split("&") if c.strip() != ""])
)

In [35]:
df.drop(columns=["Secondary Grape Varieties", "Appellation", "Unit", 'Title'], inplace=True)

In [36]:
df.head()

,Title,Price,Grape,Closure,Country,Characteristics,Type,ABV,Region,Style,Vintage
0,"The Guv'nor, Spain",13.320000,Tempranillo,Natural Cork,Spain,"{Blackberry, Blackcurrant, Vanilla}",Red,14.0,NaN,"{Rich, Juicy}",NaN
1,Bread & Butter 'Winemaker's Selection' Chardon...,21.320000,Chardonnay,Natural Cork,USA,"{Green Apple, Vanilla, Pineapple, Stone Fruit,...",White,13.5,California,"{Rich, Toasty}",2021.0
2,"Oyster Bay Sauvignon Blanc 2022, Marlborough",16.653333,Sauvignon Blanc,Screwcap,New Zealand,"{Green Apple, Lemon, Grapefruit, Grass, Gooseb...",White,13.0,Marlborough,"{Zesty, Crisp}",2022.0
3,Louis Latour Mâcon-Lugny 2021/22,23.986667,Chardonnay,Natural Cork,France,"{Lemon, Peach, Floral, Apricot}",White,13.5,Burgundy,"{Ripe, Rounded}",2022.0
4,Bread & Butter 'Winemaker's Selection' Pinot N...,21.320000,Pinot Noir,Natural Cork,USA,"{Smoke, Cedar, Raspberry, Red Fruit, Black Che...",Red,13.5,California,"{Smooth, Mellow}",2021.0


In [37]:
df.info()

<class 'pandas.DataFrame'>
Index: 1284 entries, 0 to 1289
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Title            1284 non-null   str    
 1   Price            1284 non-null   float64
 2   Grape            1275 non-null   str    
 3   Closure          1279 non-null   str    
 4   Country          1284 non-null   str    
 5   Characteristics  1284 non-null   object 
 6   Type             1279 non-null   str    
 7   ABV              1281 non-null   float64
 8   Region           1124 non-null   str    
 9   Style            1284 non-null   object 
 10  Vintage          1100 non-null   float64
dtypes: float64(3), object(2), str(6)
memory usage: 120.4+ KB


In [38]:
df.describe()

,Price,ABV,Vintage
count,1284.000000,1281.000000,1100.000000
mean,37.508332,13.420141,2019.860909
std,46.814898,1.827477,2.804799
min,4.000000,0.500000,1999.000000
25%,17.320000,12.500000,2019.000000
50%,22.653333,13.500000,2021.000000
75%,39.986667,14.000000,2022.000000
max,573.333333,40.000000,2023.000000


In [47]:
df.to_csv("../data/preprocessed/WineDatasetPAS.csv", index=False)

### Binarizing

In [41]:
def bin_numeric(series, bins, labels):
    return pd.cut(series.astype(float), bins=bins, labels=labels, include_lowest=True)

In [42]:
df["Price_bin"] = bin_numeric(
    df["Price"],
    bins=[0, 15, 25, 50, 1000],
    labels=["Price<=15", "15<Price<=25", "25<Price<=50", "Price>50"]
)

In [43]:
df["ABV_bin"] = bin_numeric(
    df["ABV"],
    bins=[0, 12.5, 13.5, 14.5, 20],
    labels=["ABV<=12.5", "12.5<ABV<=13.5", "13.5<ABV<=14.5", "ABV>14.5"]
)

In [46]:
df["Vintage_bin"] = bin_numeric(
    df["Vintage"],
    bins=[0, 2015, 2020, 2030],
    labels=["Vintage<=2015", "2015<Vintage<=2020", "Vintage>2020"]
)

In [49]:
df.drop(columns=["Price", "ABV", "Vintage"], inplace=True)

In [50]:
df.head()

,Title,Grape,Closure,Country,Characteristics,Type,Region,Style,Price_bin,ABV_bin,Vintage_bin
0,"The Guv'nor, Spain",Tempranillo,Natural Cork,Spain,"{Blackberry, Blackcurrant, Vanilla}",Red,NaN,"{Rich, Juicy}",Price<=15,13.5<ABV<=14.5,NaN
1,Bread & Butter 'Winemaker's Selection' Chardon...,Chardonnay,Natural Cork,USA,"{Green Apple, Vanilla, Pineapple, Stone Fruit,...",White,California,"{Rich, Toasty}",15<Price<=25,12.5<ABV<=13.5,Vintage>2020
2,"Oyster Bay Sauvignon Blanc 2022, Marlborough",Sauvignon Blanc,Screwcap,New Zealand,"{Green Apple, Lemon, Grapefruit, Grass, Gooseb...",White,Marlborough,"{Zesty, Crisp}",15<Price<=25,12.5<ABV<=13.5,Vintage>2020
3,Louis Latour Mâcon-Lugny 2021/22,Chardonnay,Natural Cork,France,"{Lemon, Peach, Floral, Apricot}",White,Burgundy,"{Ripe, Rounded}",15<Price<=25,12.5<ABV<=13.5,Vintage>2020
4,Bread & Butter 'Winemaker's Selection' Pinot N...,Pinot Noir,Natural Cork,USA,"{Smoke, Cedar, Raspberry, Red Fruit, Black Che...",Red,California,"{Smooth, Mellow}",15<Price<=25,12.5<ABV<=13.5,Vintage>2020


In [51]:
df.drop(columns=["Title"], inplace=True)

In [52]:
import pandas as pd
import ast
from sklearn.preprocessing import MultiLabelBinarizer

# Example: df = your dataframe

# 1️⃣ Identify multi-label columns (columns storing sets/lists)
multi_label_cols = ["Characteristics", "Style"]

# 2️⃣ Convert string representations of sets to actual Python lists
for col in multi_label_cols:
    df[col] = df[col].apply(
        lambda x: list(ast.literal_eval(x)) if isinstance(x, str) else list(x)
    )

# 3️⃣ One-hot encode multi-label columns
mlb_encoded_dfs = []

for col in multi_label_cols:
    mlb = MultiLabelBinarizer()
    transformed = mlb.fit_transform(df[col])

    mlb_df = pd.DataFrame(
        transformed,
        columns=[f"{col}_{c}" for c in mlb.classes_],
        index=df.index
    )

    mlb_encoded_dfs.append(mlb_df)

# 4️⃣ Drop original multi-label columns
df = df.drop(columns=multi_label_cols)

# 5️⃣ One-hot encode remaining categorical columns
df = pd.get_dummies(df, drop_first=False)

# 6️⃣ Concatenate everything
final_df = pd.concat([df] + mlb_encoded_dfs, axis=1)

   Grape_Agiorgitiko  Grape_Aglianico  Grape_Airen  Grape_Albarino  \
0              False            False        False           False   
1              False            False        False           False   
2              False            False        False           False   
3              False            False        False           False   
4              False            False        False           False   

   Grape_Alicante Bouschet  Grape_Aligoté  Grape_Alvarinho  Grape_Arinto  \
0                    False          False            False         False   
1                    False          False            False         False   
2                    False          False            False         False   
3                    False          False            False         False   
4                    False          False            False         False   

   Grape_Assyrtiko  Grape_Bacchus  ...  Style_Rich  Style_Ripe  Style_Rounded  \
0            False          False  ...   

In [53]:
final_df.head()

,Grape_Agiorgitiko,Grape_Aglianico,Grape_Airen,Grape_Albarino,Grape_Alicante Bouschet,Grape_Aligoté,Grape_Alvarinho,Grape_Arinto,Grape_Assyrtiko,Grape_Bacchus,...,Style_Rich,Style_Ripe,Style_Rounded,Style_Savoury,Style_Smooth,Style_Soft,Style_Spicy,Style_Toasty,Style_Zesty,Style_nan
0,False,False,False,False,False,False,False,False,False,False,...,1,0,0,0,0,0,0,0,0,0
1,False,False,False,False,False,False,False,False,False,False,...,1,0,0,0,0,0,0,1,0,0
2,False,False,False,False,False,False,False,False,False,False,...,0,0,0,0,0,0,0,0,1,0
3,False,False,False,False,False,False,False,False,False,False,...,0,1,1,0,0,0,0,0,0,0
4,False,False,False,False,False,False,False,False,False,False,...,0,0,0,0,1,0,0,0,0,0


In [54]:
final_df.info()

<class 'pandas.DataFrame'>
Index: 1284 entries, 0 to 1289
Columns: 390 entries, Grape_Agiorgitiko to Style_nan
dtypes: bool(253), int64(137)
memory usage: 1.7 MB


In [56]:
final_df.to_csv("../data/binarized/WineDatasetBIN.csv", index=False)